# `order_book` Python API demo

This notebook demonstrates:

1. That the `order-book` package can be `pip install`ed into a virtual environment (the C++ engine gets compiled for you via [scikit-build-core](https://scikit-build-core.readthedocs.io/) — no separate CMake invocation needed).
2. How to use the resulting Python API: building an order book, submitting the different order types, feed-driven cancels/reduces/executes, replaying historical ITCH data through `SimulationEngine`, and computing microstructure features from a snapshot.

See the main [README](../README.md#python-library) for the full reference.

## 1. Set up an isolated environment

Run these commands in a terminal **before** launching this notebook (skip if you already have a venv with Jupyter set up how you like):

```bash
cd order-book               # repo root
python3 -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install jupyter ipykernel
python -m ipykernel install --user --name order-book-demo
jupyter notebook python/order_book_demo.ipynb   # then select the "order-book-demo" kernel
```

Building the extension requires a C++17 compiler and CMake >= 3.15 on your `PATH` (pip fetches pybind11 itself via CMake's `FetchContent` — no manual pybind11 install needed).

The cell below just confirms which interpreter this notebook's kernel is actually using, so you can verify it's the venv you created.

In [3]:
import sys

print('Python executable:', sys.executable)
print('Running inside a virtual environment:', sys.prefix != sys.base_prefix)

Python executable: /home/az721/projects/order-book/python/.venv/bin/python
Running inside a virtual environment: True


## 2. Install `order-book` into this environment

This is the actual proof: running `pip install` against the repo root (one directory up from this notebook) and confirming the package imports afterward. The first install compiles the C++ engine, so this can take ~30-60s.

In [4]:
%pip install -q ..

Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys
import sysconfig

# This notebook's own directory (python/) also contains order_book's SOURCE
# package (without the compiled extension) — Jupyter puts that directory on
# sys.path, which would otherwise shadow the properly pip-installed copy the
# cell above just built. Force the installed site-packages copy to win.
sys.path.insert(0, sysconfig.get_paths()['purelib'])

import order_book as ob

print('order_book package loaded from:', ob.__file__)
print('compiled extension loaded from:', ob._order_book.__file__)

order_book package loaded from: /home/az721/projects/order-book/python/.venv/lib/python3.14/site-packages/order_book/__init__.py
compiled extension loaded from: /home/az721/projects/order-book/python/.venv/lib/python3.14/site-packages/order_book/_order_book.cpython-314-aarch64-linux-gnu.so


## 3. Core types: enums and price utilities

Prices throughout the engine are `int64` **basis points** (1 unit = $0.0001) rather than floats, to avoid floating-point rounding errors in matching logic. `to_basis_points` / `from_basis_points` / `format_price` convert to and from ordinary dollar amounts.

In [6]:
print(ob.Side.BID, ob.Side.ASK)
print(ob.OrderType.LIMIT, ob.OrderType.MARKET, ob.OrderType.IMMEDIATE_OR_CANCEL, ob.OrderType.FILL_OR_KILL)
print('opposite of BID:', ob.opposite(ob.Side.BID))

price_bp = ob.to_basis_points(150.05)
print()
print('150.05 dollars ->', price_bp, 'basis points')
print('back to dollars:', ob.from_basis_points(price_bp))
print('formatted:', ob.format_price(price_bp))

Side.BID Side.ASK
OrderType.LIMIT OrderType.MARKET OrderType.IMMEDIATE_OR_CANCEL OrderType.FILL_OR_KILL
opposite of BID: Side.ASK

150.05 dollars -> 1500500 basis points
back to dollars: 150.05
formatted: 150.0500


## 4. Building an order book and submitting orders

`TreeOrderBook` is the MVP book implementation (red-black tree of price levels, FIFO price-time priority via `FIFOMatcher`). Callbacks fire synchronously as orders are processed.

In [7]:
fills = []
acks = []

callbacks = ob.OrderBookCallbacks()
callbacks.on_fill = lambda f: fills.append(f)
callbacks.on_ack = lambda a: acks.append(a)

book = ob.TreeOrderBook('AAPL', ob.FIFOMatcher(), callbacks)

book.add(ob.Order(id=1, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.LIMIT,
                  price=ob.to_basis_points(150.00), quantity=100, timestamp=1))
book.add(ob.Order(id=2, symbol='AAPL', side=ob.Side.ASK, type=ob.OrderType.LIMIT,
                  price=ob.to_basis_points(150.05), quantity=80, timestamp=2))
book.add(ob.Order(id=3, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.LIMIT,
                  price=ob.to_basis_points(150.05), quantity=30, timestamp=3))  # crosses order 2

print(f'{len(fills)} fill(s), {len(acks)} ack(s)')
for f in fills:
    print(f'  fill: {f.fill_quantity} @ {ob.format_price(f.fill_price)} (passive order {f.passive_order_id})')

print()
print('best_bid:', ob.format_price(book.best_bid()))
print('best_ask:', book.best_ask())
print('total_bid_qty:', book.total_bid_qty())
print('total_ask_qty:', book.total_ask_qty())

1 fill(s), 3 ack(s)
  fill: 30 @ 150.0500 (passive order 2)

best_bid: 150.0000
best_ask: 1500500
total_bid_qty: 100
total_ask_qty: 50


In [8]:
snap = book.snapshot()

print('Bids:')
for level in snap.bids:
    print(f'  {ob.format_price(level.price)}  qty={level.quantity}  orders={level.order_count}')

print('Asks:')
for level in snap.asks:
    print(f'  {ob.format_price(level.price)}  qty={level.quantity}  orders={level.order_count}')

Bids:
  150.0000  qty=100  orders=1
Asks:
  150.0500  qty=50  orders=1


## 5. Order types: IOC and FOK

- `IMMEDIATE_OR_CANCEL`: fills whatever it can immediately, cancels the unfilled remainder, never rests.
- `FILL_OR_KILL`: must fill completely immediately or the whole order is rejected outright — no partial fills are ever produced.

In [9]:
book2 = ob.TreeOrderBook('AAPL', ob.FIFOMatcher())
book2.add(ob.Order(id=1, symbol='AAPL', side=ob.Side.ASK, type=ob.OrderType.LIMIT,
                   price=ob.to_basis_points(150.00), quantity=50, timestamp=1))

ioc_fills = []
cbs = ob.OrderBookCallbacks()
cbs.on_fill = lambda f: ioc_fills.append(f)
book2.set_callbacks(cbs)

book2.add(ob.Order(id=2, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.IMMEDIATE_OR_CANCEL,
                   price=ob.to_basis_points(150.00), quantity=80, timestamp=2))
print('IOC: filled', sum(f.fill_quantity for f in ioc_fills), 'of 80 requested; resting afterward:', book2.has_order(2))

book2.add(ob.Order(id=3, symbol='AAPL', side=ob.Side.ASK, type=ob.OrderType.LIMIT,
                   price=ob.to_basis_points(150.00), quantity=50, timestamp=3))

fok_fills = []
cbs2 = ob.OrderBookCallbacks()
cbs2.on_fill = lambda f: fok_fills.append(f)
book2.set_callbacks(cbs2)

book2.add(ob.Order(id=4, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.FILL_OR_KILL,
                   price=ob.to_basis_points(150.00), quantity=999, timestamp=4))
print('FOK for 999 (book only has 50): fills =', len(fok_fills), '- rejected outright, resting:', book2.has_order(4))

IOC: filled 50 of 80 requested; resting afterward: False
FOK for 999 (book only has 50): fills = 0 - rejected outright, resting: False


## 6. Cancelling, reducing, and feed-driven executes

Three distinct ways a resting order's quantity can change, each firing a different event:

- `reduce()` — a size decrease with **no trade** (e.g. a participant shrinking their own resting order). Fires `on_cancel`, keeps its place in the time-priority queue.
- `execute()` — a feed-driven **fill** against a resting order (e.g. replaying an ITCH Executed message). Fires `on_fill`.
- `cancel()` — full removal.

In [10]:
book3 = ob.TreeOrderBook('AAPL', ob.FIFOMatcher())
book3.add(ob.Order(id=1, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.LIMIT,
                   price=ob.to_basis_points(150.00), quantity=100, timestamp=1))
print('before:            ', book3.total_bid_qty())

book3.reduce(order_id=1, quantity=20, timestamp=2)
print('after reduce(20):  ', book3.total_bid_qty(), '- still resting:', book3.has_order(1))

book3.execute(order_id=1, quantity=30, timestamp=3)
print('after execute(30): ', book3.total_bid_qty(), '- still resting:', book3.has_order(1))

book3.cancel(1)
print('after cancel():    still resting:', book3.has_order(1))

before:             100
after reduce(20):   80 - still resting: True
after execute(30):  50 - still resting: True
after cancel():    still resting: False


## 7. Replaying historical data through `SimulationEngine`

`HistoricalReplayer` reads NASDAQ ITCH 5.0 files (the flat, length-prefixed framing used by NASDAQ's downloadable samples) and feeds them through `SimulationEngine` in timestamp order. To keep this notebook self-contained we hand-encode a tiny ITCH byte stream here instead of downloading a real sample file — swap `itch_path` for a real `.NASDAQ_ITCH50` file to replay actual data.

In [11]:
import struct

def u16(v): return struct.pack('>H', v)
def u32(v): return struct.pack('>I', v)
def u48(v): return v.to_bytes(6, 'big')
def u64(v): return struct.pack('>Q', v)
def stock(s): return s.encode().ljust(8)

def framed(payload):
    return u16(len(payload)) + payload

def itch_add(ref, side, shares, symbol, price_bp, ts, locate=1):
    return b'A' + u16(locate) + u16(0) + u48(ts) + u64(ref) + side.encode() + u32(shares) + stock(symbol) + u32(price_bp)

def itch_execute(ref, shares, ts, locate=1):
    return b'E' + u16(locate) + u16(0) + u48(ts) + u64(ref) + u32(shares) + u64(0)

itch_bytes = (
    framed(itch_add(1, 'B', 100, 'AAPL', ob.to_basis_points(150.00), 1_000)) +
    framed(itch_add(2, 'S', 40, 'AAPL', ob.to_basis_points(150.05), 2_000)) +
    framed(itch_execute(1, 30, 3_000))
)

import pathlib, tempfile
itch_path = pathlib.Path(tempfile.mkdtemp()) / 'sample.itch'
itch_path.write_bytes(itch_bytes)

109

In [12]:
config = ob.SimulationConfig()
config.symbol = 'AAPL'

replay_fills = []
engine = ob.SimulationEngine(config, ob.TreeOrderBook('AAPL', ob.FIFOMatcher()))
engine.set_fill_callback(lambda f: replay_fills.append(f))

replay_config = ob.HistoricalReplayerConfig()
replay_config.path = str(itch_path)
engine.add_source(ob.HistoricalReplayer(replay_config))

engine.run()

# Note: the HistoricalReplayer we constructed above transferred ownership to
# the engine (unique_ptr semantics, same as the C++ API) — use the engine's
# own accessors afterward, not that original object.
replayer = engine.historical_replayer()

print('messages parsed: ', replayer.messages_parsed())
print('orders replayed: ', replayer.orders_replayed())
print('fills:           ', [(f.passive_order_id, f.fill_quantity, ob.format_price(f.fill_price)) for f in replay_fills])
print('best_bid:        ', ob.format_price(engine.book().best_bid()))
print('total_bid_qty:   ', engine.book().total_bid_qty())

messages parsed:  3
orders replayed:  3
fills:            [(1, 30, '150.0000')]
best_bid:         150.0000
total_bid_qty:    70


## 8. Microstructure features

`order_book.features.microstructure` computes standard order-book statistics directly from a `BookSnapshot` / `FillEvent` — no live book or engine reference needed, so these work equally well on a snapshot captured live, replayed from history, or logged by a separate backtesting project.

In [13]:
from order_book.features import microstructure as ms

snap2 = engine.book().snapshot()

print('mid_price:             ', ms.mid_price(snap2))
print('micro_price:            ', ms.micro_price(snap2))
print('spread:                 ', ms.spread(snap2))
print('relative_spread:        ', ms.relative_spread(snap2))
print('order_book_imbalance:   ', ms.order_book_imbalance(snap2, max_levels=1))

if snap2.asks:
    print('price_impact (buy 20):  ', ms.price_impact(snap2, ob.Side.BID, 20))

mid_price:              1500250.0
micro_price:             1500318.1818181819
spread:                  500
relative_spread:         0.0003332777870354941
order_book_imbalance:    0.2727272727272727
price_impact (buy 20):   0.0


In [14]:
# effective_spread needs the mid price at the moment of the trade, since a
# FillEvent alone carries no book context - capture it just before the
# crossing order arrives.
book4 = ob.TreeOrderBook('AAPL', ob.FIFOMatcher())
book4.add(ob.Order(id=1, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.LIMIT,
                   price=ob.to_basis_points(150.00), quantity=100, timestamp=1))
book4.add(ob.Order(id=2, symbol='AAPL', side=ob.Side.ASK, type=ob.OrderType.LIMIT,
                   price=ob.to_basis_points(150.05), quantity=40, timestamp=2))

mid_before_trade = ms.mid_price(book4.snapshot())

trade_fills = []
cbs4 = ob.OrderBookCallbacks()
cbs4.on_fill = lambda f: trade_fills.append(f)
book4.set_callbacks(cbs4)
book4.add(ob.Order(id=3, symbol='AAPL', side=ob.Side.BID, type=ob.OrderType.LIMIT,
                   price=ob.to_basis_points(150.05), quantity=10, timestamp=3))

print('mid price just before the trade:', mid_before_trade)
print('fill price:                     ', ob.format_price(trade_fills[0].fill_price))
print('effective_spread:               ', ms.effective_spread(trade_fills[0], mid_before_trade))

mid price just before the trade: 1500250.0
fill price:                      150.0500
effective_spread:                500.0


Full Python API reference: the main [README](../README.md#python-library).